# 4부. 머신러닝 모델링과 평가

장별 완성 예제. 학습용 가상 데이터이며 공식 시험 제출 형식이 아니다. examples 폴더에서 전체 실행한다.

In [ ]:
from pathlib import Path
import os
if not Path('lab.py').is_file() and Path('../lab.py').is_file():
    os.chdir('..')
assert Path('lab.py').is_file(), 'Run with the examples folder and data present'
get_ipython().run_line_magic('matplotlib', 'inline')


## 16장. 머신러닝 학습의 기본 흐름

문제 유형·기준 모델·학습·예측의 순서를 익힌다.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
baseline = DummyClassifier(strategy="most_frequent").fit(X_train,y_train)
model = Pipeline([("prep",preprocessor()),
                  ("clf",LogisticRegression(max_iter=1000))]).fit(X_train,y_train)
baseline_score = accuracy_score(y_val,baseline.predict(X_val))
model_score = accuracy_score(y_val,model.predict(X_val))
assert 0 <= baseline_score <= 1 and 0 <= model_score <= 1
print({"baseline_accuracy":baseline_score, "model_accuracy":model_score})

## 17장. 분류 모델 만들기

로지스틱 회귀와 의사결정나무를 학습한다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
models = {"logistic":LogisticRegression(max_iter=1000),
          "tree":DecisionTreeClassifier(max_depth=4,random_state=42)}
for name, estimator in models.items():
    model = Pipeline([("prep",preprocessor()),("clf",estimator)])
    model.fit(X_train,y_train)
    prediction = model.predict(X_val)
    print(name, accuracy_score(y_val,prediction), recall_score(y_val,prediction))
    assert set(prediction).issubset({0,1})

## 18장. 회귀 모델 만들기

선형 회귀와 트리 회귀의 예측을 비교한다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task("regression")
models = {"linear":LinearRegression(),
          "tree":DecisionTreeRegressor(max_depth=5,random_state=42)}
for name, estimator in models.items():
    model = Pipeline([("prep",preprocessor("regression")),("reg",estimator)])
    model.fit(X_train,y_train)
    prediction = model.predict(X_val)
    assert prediction.shape == y_val.shape
    print(name, "MAE:",mean_absolute_error(y_val,prediction),
          "RMSE:",root_mean_squared_error(y_val,prediction))

## 19장. 앙상블 모델 활용하기

랜덤 포레스트와 부스팅의 설정과 결과를 비교한다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
estimators = {
    "forest":RandomForestClassifier(n_estimators=80,max_depth=6,random_state=42,n_jobs=1),
    "boosting":GradientBoostingClassifier(n_estimators=60,learning_rate=0.05,max_depth=2,random_state=42),
}
for name, estimator in estimators.items():
    model = Pipeline([("prep",preprocessor()),("clf",estimator)]).fit(X_train,y_train)
    prediction = model.predict(X_val)
    value = f1_score(y_val,prediction,zero_division=0)
    assert 0 <= value <= 1
    print(name, "validation F1:",value)

## 20장. 과적합을 확인하고 성능 개선하기

훈련·검증 차이·교차검증·설정 탐색을 이해한다.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
pipeline = Pipeline([("prep",preprocessor()),
                     ("clf",DecisionTreeClassifier(random_state=42))])
cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
search = GridSearchCV(pipeline, {"clf__max_depth":[2,4,6]},
                      scoring="f1",cv=cv,n_jobs=1)
search.fit(X_train,y_train)
best = search.best_estimator_
print(search.best_params_, search.best_score_)
print("train accuracy:",accuracy_score(y_train,best.predict(X_train)))
print("validation accuracy:",accuracy_score(y_val,best.predict(X_val)))
assert search.best_params_["clf__max_depth"] in [2,4,6]

## 21장. 모델 성능을 정확히 평가하기

분류·회귀 지표와 그래프를 문제 목적에 연결한다.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
    roc_auc_score, ConfusionMatrixDisplay)
from lab import split_task, preprocessor

X_train, X_val, X_test, y_train, y_val, y_test = split_task()
model = Pipeline([("prep",preprocessor()),("clf",LogisticRegression(max_iter=1000))])
model.fit(X_train,y_train)
pred = model.predict(X_val)
classes = list(model.named_steps["clf"].classes_)
prob = model.predict_proba(X_val)[:,classes.index(1)]
matrix = confusion_matrix(y_val,pred,labels=[0,1])
assert matrix.sum() == len(y_val)
print(classification_report(y_val,pred,zero_division=0))
print("ROC AUC:",roc_auc_score(y_val,prob))
ConfusionMatrixDisplay(matrix,display_labels=[0,1]).plot()
plt.show()
plt.close("all")